# Homework 3

by: Najla Tsabita Afiyah

In [91]:
import pandas as pd
import numpy as np

In [92]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [93]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [94]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [95]:
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].fillna('NA')

for col in df.select_dtypes(include=['number']).columns:
    df[col] = df[col].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

### Q1. Mode for industry
What is the most frequent observation (mode) for the column industry?

In [96]:
df['industry'].mode()[0]

'technology'

### Q2. Biggest correlation
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

In [97]:
numerical_cols = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
corr_matrix = df[numerical_cols].corr()
print(corr_matrix)

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  


#### Split the data

In [98]:
from sklearn.model_selection import train_test_split

In [99]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

### Q3. Biggest MI 
- Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using round(score, 2).
Which of these variables has the biggest mutual information score?

In [100]:
from sklearn.metrics import mutual_info_score

In [101]:
categorical_cols = ['lead_source', 'industry', 'employment_status', 'location']

y_train = df_train.converted.values

mi_scores = df_train[categorical_cols].apply(
    lambda col: round(mutual_info_score(col, y_train), 2)
)
mi_scores.sort_values(ascending=False)

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64

### Q4. Accuracy 
- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
    - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
    - model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
What accuracy did you get?

In [102]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [103]:
numerical_cols = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
categorical_cols = ['lead_source', 'industry', 'employment_status', 'location']
features = categorical_cols + numerical_cols
print(features)

['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']


In [104]:
y_train = df_train.converted.values
y_val = df_val.converted.values

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records'))

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
print(accuracy, round(accuracy, 2))

0.645 0.65


### Q5. Feature selection
- Let's find the least useful feature using the feature elimination technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
Which of following feature has the smallest difference?

In [105]:
def train_and_eval(feature_list):
    dv = DictVectorizer(sparse=False)
    X_tr = dv.fit_transform(df_train[feature_list].to_dict(orient='records'))
    X_va = dv.transform(df_val[feature_list].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_tr, y_train)

    return accuracy_score(y_val, model.predict(X_va))

features = categorical_cols + numerical_cols
base_acc = train_and_eval(features)
print("initial accuracy:", base_acc)

results = []
for f in features:
    reduced = [x for x in features if x != f]
    acc = train_and_eval(reduced)
    results.append((f, acc, base_acc - acc))

df_fe = pd.DataFrame(results, columns=['feature', 'acc_without', 'diff'])
df_fe.sort_values('diff')

initial accuracy: 0.645


,feature,acc_without,diff
4,annual_income,0.724,-0.079
1,industry,0.645,0.000
3,location,0.645,0.000
2,employment_status,0.645,0.000
7,lead_score,0.644,0.001
5,number_of_courses_viewed,0.643,0.002
0,lead_source,0.642,0.003
6,interaction_count,0.601,0.044


### Q6. Parameter tuning
- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
Which of these C leads to the best accuracy on the validation set?

In [106]:
C_values = [0.000001, 0.00001, 0.0001, 0.001]

for C in C_values:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    print(f"C={C}: {round(acc, 3)}")

C=1e-06: 0.598
C=1e-05: 0.598
C=0.0001: 0.613
C=0.001: 0.645
